# 01 · Data acquisition & exploratory data analysis
**Dataset:** UCI Heart Disease (Cleveland subset, 303 patients, 13 clinical features + diagnosis `num`).  
**Goal of this notebook:** understand data quality, class balance and which signals separate patients with and without heart disease — *before* choosing models.

Reproduce the data with `python scripts/download_data.py --prepare`.

In [ ]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))   # works even if `pip install -e .` was skipped

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
from heartrisk import config
from heartrisk.data import read_raw, clean, quality_report
from heartrisk import eda
eda._style()

## 1 · Raw data and cleaning decisions

In [ ]:
raw = read_raw()
print("raw shape:", raw.shape)
raw.head()

In [ ]:
df = clean(raw)
print(f"clean shape: {df.shape}")
display(pd.Series(quality_report(df)["missing_by_column"], name="missing cells").to_frame())
df.describe().T.round(2)

**Cleaning decisions**
* `?` → `NaN`; all columns coerced to numeric; exact duplicate rows removed.
* The original label `num` (0–4 = severity) is collapsed to a binary `target` (1 = any disease), as required by the problem statement.
* Only `ca` (4 cells) and `thal` (2 cells) are missing. They are **kept as NaN here and imputed inside the sklearn Pipeline** (median / mode, fitted on training folds only) so no information from validation or test data leaks into training.

## 2 · Class balance

In [ ]:
display(Image(str(eda.class_balance(df, config.FIGURES_DIR))))

The classes are close to balanced (≈46 % disease), so accuracy is meaningful, but we still report precision, recall and ROC-AUC. In a screening context **recall** (not missing sick patients) deserves particular attention.

## 3 · Distributions

In [ ]:
display(Image(str(eda.numeric_histograms(df, config.FIGURES_DIR))))
display(Image(str(eda.boxplots(df, config.FIGURES_DIR))))

Disease patients tend to show a **lower maximum heart rate (`thalach`)**, **higher ST depression (`oldpeak`)** and more coloured vessels (`ca`). Cholesterol and resting BP overlap heavily and carry less signal on their own. A few cholesterol outliers exist, but they are physiologically plausible so they are retained (the scaler is fitted on training data only).

## 4 · Categorical & binary features vs outcome

In [ ]:
display(Image(str(eda.categorical_vs_target(df, config.FIGURES_DIR))))

Asymptomatic chest pain (`cp = 4`), exercise-induced angina, reversible-defect thalassemia (`thal = 7`) and flat/down slopes are strongly associated with disease — useful hints for feature selection and for sanity-checking model explanations later.

## 5 · Correlations

In [ ]:
display(Image(str(eda.correlation_heatmap(df, config.FIGURES_DIR))))
display(Image(str(eda.target_correlations(df, config.FIGURES_DIR))))

No pair of predictors is highly collinear (|r| < 0.6), so a regularised linear model is well-conditioned. This motivates engineering a few *non-linear / threshold* clinical features (`hr_pct_max`, `high_bp`, `high_chol`, `st_depressed`) in `heartrisk/features.py`.

In [ ]:
eda.run_eda(df, config.FIGURES_DIR)   # writes all figures + reports/eda_summary.md
print("Figures saved in", config.FIGURES_DIR)